# Pretrained models

Let's test a few pretrained models -- there are definitely others you can try!

**CheMeleon** is a Chemprop encoder pre-trained on a large molecular corpus. You
fine-tune it on your data instead of starting from random weights. At least eight
of the top twenty finishers in the real challenge used it.

**TabPFN** is a transformer pre-trained on millions of synthetic tabular
problems. It does not train on your data at all &mdash; you hand it your table at
inference time and it predicts.

Everything here is **single-task**: one model per endpoint. That is deliberate.
Multitask learning is a separate idea with its own card, and mixing the two makes
it impossible to tell which one earned you the improvement.

---
### Setup

In [ ]:
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec lightgbm matplotlib seaborn pyarrow chemprop==2.3.1 tabpfn==8.2.0
!git clone https://github.com/agura-alt/ai4chem_openadmet.git
%cd ai4chem_openadmet

In [ ]:
#@title Imports...

import os, sys
SETUP_DIR = os.path.abspath("Setup")
os.path.isdir(SETUP_DIR) or sys.exit(f"No Setup dir at {SETUP_DIR}; cwd is {os.getcwd()}")

if SETUP_DIR not in sys.path:
    sys.path.insert(0, SETUP_DIR)

assert os.path.exists("Setup/common.py") and os.path.getsize("Setup/common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)            # force a fresh read
import common

Change `your-pair-name` to your team name. It has to match the list of registered teams exactly, and be the same in every notebook &mdash; that is what links your work together.

In [ ]:
# Same folder as every other notebook -- splits, predictions, scores.
common.setup(pair="your-pair-name")

Check your runtime!

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\n>>> Runtime -> Change runtime type -> T4 GPU, then re-run. <<<")
    print(">>> TabPFN will still work on CPU, just slowly. CheMeleon needs the GPU. <<<")

GPU available: True


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os, subprocess
import glob

sns.set_style("whitegrid")
WORK = common.workdir()

train = common.load_train()
test  = common.load_test()

display(common.list_splits())

,name,method,n_train,n_val,saved_at
0,random,random,4261,1065,2026-08-14T08:22:59+00:00
1,similarity,similarity,4261,1065,2026-08-14T08:22:59+00:00
2,temporal,temporal,4261,1065,2026-08-14T08:22:59+00:00


In [ ]:
SPLIT = "random"        # <-- change to whichever you trust
fold, split_meta = common.load_split(train, name=SPLIT)

train_df = train[(fold == "train").to_numpy()].reset_index(drop=True)
val_df   = train[(fold == "val").to_numpy()].reset_index(drop=True)
print(f"{len(train_df)} train / {len(val_df)} val molecules")

# How much data each endpoint actually has
counts = train_df[common.ENDPOINTS].notna().sum().sort_values()
print("\nmeasurements per endpoint in your training fold:")
print(counts.to_string())

using your saved split 'random' (1065 val molecules)
4261 train / 1065 val molecules

measurements per endpoint in your training fold:
Log_Mouse_MPB        182
Log_Mouse_BPB        780
Log_Mouse_PPB       1050
Log_Caco_Papp_AB    1724
Log_Caco_ER         1728
Log_HLM_CLint       2998
Log_MLM_CLint       3613
LogD                4030
LogS                4104


---
## 1. CheMeleon, one endpoint at a time

Pre-training is supposed to help most where you have least data. So rather than
running all nine endpoints, pick one with plenty and one with very little, and
see whether the gap between from-scratch and fine-tuned differs.

**Before you run anything: where do you expect pre-training to help more,
and by how much?** Which endpoints will benefit more and which not so much?

In [ ]:
#@title Chemeleon helpers...

def write_csv(df, path, endpoint):
    """Chemprop reads a CSV of SMILES plus target columns. One target here."""
    out = df[["SMILES", endpoint]].dropna(subset=[endpoint])
    out.to_csv(path, index=False)
    return path, len(out)


def run_chemprop_cmd(cmd):
    """Run a chemprop command and fail readably if the CLI has moved."""
    done = subprocess.run(cmd, capture_output=True, text=True)
    if done.returncode != 0:
        tail = (done.stderr or done.stdout or "").strip().split("\n")[-6:]
        raise RuntimeError(
            "chemprop failed:\n  " + " ".join(cmd[:3]) + " ...\n  "
            + "\n  ".join(tail)
            + "\n\nIf it says 'unrecognized arguments', the CLI changed between\n"
              "versions. Check `!chemprop train --help` and fix the flags below.")
    return done

_TRAIN_HELP = {}

def chemprop_train_flags():
    """The flags THIS chemprop accepts, so we can check one before relying on it."""
    if "text" not in _TRAIN_HELP:
        done = subprocess.run(["chemprop", "train", "--help"],
                              capture_output=True, text=True)
        _TRAIN_HELP["text"] = (done.stdout or "") + (done.stderr or "")
    return _TRAIN_HELP["text"]


def find_checkpoint(model_dir):
    """Where chemprop left the trained model.

    The layout has moved between 2.x releases, so look for the file rather
    than assuming model_0/best.pt.
    """
    best = os.path.join(model_dir, "model_0", "best.pt")
    if os.path.exists(best):                    # what a finished run writes
        return best
    for pattern in ("model_*/best.pt", "model_*/checkpoints/*.ckpt",
                    "**/best.pt", "**/*.ckpt"):
        hits = sorted(glob.glob(os.path.join(model_dir, pattern), recursive=True))
        if hits:
            # No best.pt means training did not reach the end -- say so rather
            # than quietly predicting with a half-trained model.
            print(f"  !! no model_0/best.pt in {model_dir} -- training may have "
                  f"been interrupted.\n     Falling back to "
                  f"{os.path.relpath(hits[0], model_dir)}, which may be "
                  "partially trained.")
            return hits[0]
    raise FileNotFoundError(
        f"No trained model file under {model_dir}.\n"
        f"  it contains: {sorted(os.listdir(model_dir))}\n"
        "Training probably did not finish -- re-run the training cell and read "
        "its output.")


def run_single(endpoint, tag, training_data, epochs=30, foundation=None):
    """Train a SINGLE-TASK Chemprop model on one endpoint.

    training_data : the frame to train on. Pass train_df while experimenting,
                    the full `train` for the model you submit.
    """
    run_dir = os.path.join(WORK, "chemprop", tag)
    os.makedirs(run_dir, exist_ok=True)

    # chemprop takes its early-stopping slice out of what you give it, so your
    # validation fold is never seen during training.
    train_path, n_train = write_csv(training_data,
                                    os.path.join(run_dir, "train.csv"), endpoint)

    cmd = ["chemprop", "train",
           "--data-path", train_path,
           "--split-sizes", "0.9", "0.1", "0.0",   # train / early-stop / unused
           "--task-type", "regression",
           "--smiles-columns", "SMILES",
           "--target-columns", endpoint,
           "--output-dir", run_dir,
           "--epochs", str(epochs),
           "--num-workers", "0"]
    if foundation:
        # The flag moved around in chemprop 2.x, so check for it rather than
        # spending ten minutes on a run that never loaded the pretrained weights.
        # See https://github.com/JacksonBurns/chemeleon
        if "--from-foundation" not in chemprop_train_flags():
            raise RuntimeError(
                "This chemprop build has no --from-foundation flag, so it "
                "cannot load CheMeleon.\nRun `!chemprop train --help`, find the "
                "equivalent flag for your version, and fix run_single().")
        cmd += ["--from-foundation", foundation]
    print(f"  {tag}: training on {n_train} molecules")
    run_chemprop_cmd(cmd)
    return run_dir


def predict_single(model_dir, target_data, endpoint, tag):
    run_dir = os.path.join(model_dir, "pred_" + tag)
    os.makedirs(run_dir, exist_ok=True)
    input_path = os.path.join(run_dir, "input.csv")
    target_data[["SMILES"]].to_csv(input_path, index=False)
    output_path = os.path.join(run_dir, "preds.csv")

    run_chemprop_cmd(["chemprop", "predict",
                      "--test-path", input_path,
                      "--model-path", find_checkpoint(model_dir),
                      "--preds-path", output_path,
                      "--smiles-columns", "SMILES"])

    preds = pd.read_csv(output_path)
    if endpoint in preds.columns:
        column = endpoint
    else:
        # chemprop names the column after the target, or pred_0 -- take the last
        # numeric column, never the SMILES it echoes back.
        numeric = [c for c in preds.columns
                   if c != "SMILES" and pd.api.types.is_numeric_dtype(preds[c])]
        if not numeric:
            raise KeyError(f"chemprop wrote no prediction column for "
                           f"{endpoint!r}. It produced: {list(preds.columns)}")
        column = numeric[-1]
    return pd.DataFrame({"Molecule Name": target_data["Molecule Name"].to_numpy(),
                         endpoint: preds[column].to_numpy()})

> Each cell below trains two models. **Expect 5&ndash;10 minutes per
> endpoint on a T4.** We suggest comparing an endpoint with a lot of data to one with not as much -- though if you do have a lot more data, the model will take longer to learn.

In [ ]:
ENDPOINT = ...
results = {}

dir_scratch   = run_single(ENDPOINT, f"{ENDPOINT}_scratch", train_df, epochs=30)
dir_chemeleon = run_single(ENDPOINT, f"{ENDPOINT}_chemeleon", train_df, epochs=30,
                           foundation="CheMeleon")

for label, run_dir in [("from scratch", dir_scratch), ("from CheMeleon", dir_chemeleon)]:
    preds = predict_single(run_dir, val_df, ENDPOINT, label.replace(" ", "_"))
    model_name = f"chemprop-{ENDPOINT}-{'scratch' if 'scratch' in label else 'chemeleon'}"
    metrics = common.score(val_df, preds, model_name, SPLIT,
                           endpoints=[ENDPOINT], note=label)
    results[(ENDPOINT, label)] = metrics.loc[ENDPOINT]

pd.DataFrame(results).T.round(3)

In [ ]:
ENDPOINT = ...
results = results if "results" in globals() else {}   # keep the previous rows

dir_scratch   = run_single(ENDPOINT, f"{ENDPOINT}_scratch", train_df, epochs=30)
dir_chemeleon = run_single(ENDPOINT, f"{ENDPOINT}_chemeleon", train_df, epochs=30,
                           foundation="CheMeleon")

for label, run_dir in [("from scratch", dir_scratch), ("from CheMeleon", dir_chemeleon)]:
    preds = predict_single(run_dir, val_df, ENDPOINT, label.replace(" ", "_"))
    model_name = f"chemprop-{ENDPOINT}-{'scratch' if 'scratch' in label else 'chemeleon'}"
    metrics = common.score(val_df, preds, model_name, SPLIT,
                           endpoints=[ENDPOINT], note=label)
    results[(ENDPOINT, label)] = metrics.loc[ENDPOINT]

pd.DataFrame(results).T.round(3)

**Compare the two gaps, not the two scores.**

An endpoint with 1,000 measurements can learn a decent representation from your
data alone. One with 111 cannot, and has to borrow.

Which endpoint would you now spend your remaining GPU time on?

---
## 2. TabPFN, a model that does not train

No fitting step: you pass your training table (i.e., your descriptor dataframe and labels) in at inference time and it
predicts!

In [ ]:
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

TABPFN_VERSION = ModelVersion.V2

---
### First, decide what TabPFN gets to see
Choose your descriptors! We load **mordred** as a starting point &mdash; ~1370
physicochemical numbers per molecule, precomputed for you &mdash; but you can
choose any you like. RDKit's ~195 descriptors are one line away, and `Descriptors`
has Morgan fingerprints too.

In [ ]:
#@title mordred loader...
def load_mordred(df):
    """Mordred descriptors for `df`, one row per molecule, in df's own row order."""
    if not os.path.exists(MORDRED_PATH):
        raise FileNotFoundError(
            f"No mordred descriptors at {MORDRED_PATH}.\n"
            "The file is precomputed and committed -- check that Data/artifacts/ "
            "came down with the repo, or rebuild it with\n"
            "  python Setup/precompute_descriptors.py")
    table = pd.read_parquet(MORDRED_PATH).set_index("Molecule Name")
    # reindex, not filter: this lines each row up with df BY NAME, rather than
    # trusting the file to happen to be in the same order.
    return (table.reindex(df["Molecule Name"]).reset_index(drop=True)
                 .apply(pd.to_numeric, errors="coerce"))

In [ ]:
endpoint = ...        # select your endpoint

# Mordred: ~1370 physicochemical descriptors.
MORDRED_PATH = common.data_path(os.path.join("artifacts", "mordred_descriptors.parquet"))

X_train = load_mordred(train_df)          # select your descriptors
X_val   = load_mordred(val_df)
# ...or swap both lines for RDKit's 195:
# X_train = common.rdkit_descriptors(train_df["SMILES"])
# X_val   = common.rdkit_descriptors(val_df["SMILES"])

X_train_clean, X_val_clean = common.clean_features(X_train, X_val)
print(X_train_clean.shape, "descriptors")

# filter to only those with values for the endpoint
y_train = train_df[endpoint]
ok = y_train.notna().to_numpy()
X_endpoint, y_endpoint = X_train_clean[ok], y_train[ok]

TabPFN takes your table at inference time, and it comes with a **budget**: a limit
on rows, and a hard limit of **500 columns**. Mordred hands you around 1370.

So something has to go, and *which* columns you keep and *how many* is a modelling
decision you have to make. Questions worth answering with a run rather than a guess:

To perform the downselection, we provide a starting point of `SelectKBest` with `f_regression`: keep the `k` columns
most linearly correlated with the endpoint. There are many other ways to choose
&mdash; see the [scikit-learn feature selection
docs](https://scikit-learn.org/stable/modules/feature_selection.html).

Another approach is dimensionality reduction, e.g., using
[PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html).
Mordred columns run from 0/1 flags to molecular weights in the hundreds, and PCA
maximises variance &mdash; so what would your first component mostly be made of if
you skipped the `StandardScaler`?

All the code below that runs TabPFN expects dataframes that have reduced features!

In [ ]:
####### here comes the feature reduction! #######
from sklearn.feature_selection import SelectKBest, f_regression

MAX_FEATURES = 500       # TabPFN v2's hard limit -- it raises an error above this
K_FEATURES   = 128       # <-- your call, anywhere up to MAX_FEATURES

assert K_FEATURES <= MAX_FEATURES, (
    f"TabPFN v2 takes at most {MAX_FEATURES} features, you asked for {K_FEATURES}.")

selector = SelectKBest(f_regression, k=K_FEATURES)
selector.fit(X_endpoint, y_endpoint)
selector.set_output(transform="pandas")   # keep it a DataFrame: names survive

X_endpoint_sel = selector.transform(X_endpoint)      # what the model fits on
X_val_sel      = selector.transform(X_val_clean)     # what it predicts from

print(f"kept {X_endpoint_sel.shape[1]} of {X_endpoint.shape[1]} descriptors:")
print(", ".join(X_endpoint_sel.columns[:20]), "...")

We provide a convenience function to train TabPFN -- the initialize - fit - predict loop is very similar to scikit-learn!

In [ ]:
def run_tabpfn(X_train, y_train, max_n=3000, seed=0):
    """Hand TabPFN an ALREADY-REDUCED table of features. Returns the model.

    X_train : features, reduced however you chose above.
    y_train : the endpoint values for those same rows, in the same order.
    """
    if len(X_train) > max_n:                 # respect the sample-count limit
        sample_idx = np.random.default_rng(seed).choice(len(X_train), max_n,
                                                        replace=False)
        X_train, y_train = X_train.iloc[sample_idx], y_train.iloc[sample_idx]

    ### initialize - fit - predict
    model = TabPFNRegressor.create_default_for_version(TABPFN_VERSION)
    model.fit(X_train, y_train)
    return model

In [ ]:
model = run_tabpfn(X_endpoint_sel, y_endpoint)
print(f"TabPFN saw {min(len(X_endpoint_sel), 3000)} molecules "
      f"x {X_endpoint_sel.shape[1]} features")

In [ ]:
preds = pd.DataFrame({"Molecule Name": val_df["Molecule Name"].to_numpy()})
preds[endpoint] = model.predict(X_val_sel)

# Scoring one endpoint on its own!
metrics = common.score(val_df, preds, f"tabpfn-{endpoint}-k{K_FEATURES}", SPLIT,
                       endpoints=[endpoint], note=f"TabPFN, k={K_FEATURES}")
metrics.round(3)

In [ ]:
common.score_matrix()

- What happens to your score as you raise `k`? Is there a point where more columns
  stop earning their place?
- What does each doubling of `k` cost you in wall-clock time? With nine endpoints to
  fit, where would you stop?
- `Log_Mouse_MPB` has a couple of hundred measured molecules, and you are about to
  rank ~1370 columns against them. What could go wrong? Would you trust the same
  ranking on the validation set?
- The endpoints disagree about which columns matter. Do they disagree about how
  *many* they need, too?

This cell prepares a feature reduction using SelectKBest for every endpoint. SelectKBest is SUPERVISED -- it ranks columns against y -- so the best 32 descriptors for LogD are not the best 32 for Log_Mouse_MPB, and each endpoint gets its own selector fitted on its own measured molecules.

Swap these three selector lines for your own reduction!

In [ ]:
K_FEATURES = ...
assert K_FEATURES <= MAX_FEATURES, (
    f"TabPFN v2 takes at most {MAX_FEATURES} features, you asked for {K_FEATURES}.")

REDUCED = {}
for e in common.ENDPOINTS:
    ok = train_df[e].notna().to_numpy()
    sel = SelectKBest(f_regression, k=min(K_FEATURES, X_train_clean.shape[1]))
    sel.set_output(transform="pandas")
    sel.fit(X_train_clean[ok], train_df[e][ok])

    REDUCED[e] = (sel.transform(X_train_clean[ok]),   # train rows for THIS endpoint
                  sel.transform(X_val_clean))         # every validation molecule
    print(f"  {e:18s} {int(ok.sum()):5d} molecules -> {REDUCED[e][0].shape[1]} features")

In [ ]:
MODEL = "tabpfn-mordred"        # the name your submission will cite

rows = {}
all_preds = pd.DataFrame({"Molecule Name": val_df["Molecule Name"].to_numpy()})

for endpoint in common.ENDPOINTS:
    X_tr, X_va = REDUCED[endpoint]              # prepared above, already reduced
    y_tr = train_df[endpoint].dropna()          # same rows, same order as X_tr

    model = run_tabpfn(X_tr, y_tr)
    preds = pd.DataFrame({"Molecule Name": val_df["Molecule Name"].to_numpy()})
    preds[endpoint] = model.predict(X_va)

    all_preds[endpoint] = preds[endpoint].to_numpy()
    rows[endpoint] = common.score(val_df, preds, f"tabpfn-{endpoint}", SPLIT,
                                  endpoints=[endpoint],
                                  note=f"TabPFN, k={K_FEATURES}").loc[endpoint]

# ...and one row for the model as a whole. All nine endpoints, so no partial-model
# warning -- and this is the row prepare_submission looks for further down.
common.score(val_df, all_preds, MODEL, SPLIT, note=f"TabPFN, k={K_FEATURES}")

tabpfn_all = pd.DataFrame(rows).T
tabpfn_all.round(3)

Compare that RAE column against your LightGBM baseline from
`01_validation`. TabPFN did not top the real leaderboard, but it needed no
tuning and no training &mdash; which makes it a strong thing to have in an
ensemble.

---
## Save your work

Give it a name you will recognise! `Ensembles` can combine this with anything
else you have made today &mdash; and a pretrained model is often usefully
*different* from a gradient-boosted one, which is exactly what makes an ensemble
work.

### Make a submission
Go back and try another split, or another reduction, if you like &mdash; then make a
submission.
Note: the model name is the one the nine-endpoint cell logged &mdash; `tabpfn-mordred`
for the model as a whole, or `tabpfn-'endpoint'` for the per-endpoint rows.

In [ ]:
common.score_matrix()

** Note: ** The cell below fits one TabPFN per endpoint on every labelled
molecule, so the submission is TabPFN all the way across. If you want a mix
&mdash; CheMeleon for the endpoints where it won, something cheaper elsewhere
&mdash; you will have to write your own prediction saver, and you should score
that same combination with `common.score` rather than borrowing a number from
either model alone. `Ensembles` has `per_endpoint_best` for exactly this.

In [ ]:
# The model you submit trains on EVERY labelled molecule -- pass `train` rather
# than train_df. The split above was only for estimating the score.

MODEL = ...
SPLIT = ...

# Descriptors for the full training set and the test set, cleaned together so the
# selector's columns line up across both.
X_full, X_test_clean = common.clean_features(load_mordred(train), load_mordred(test))

test_preds = pd.DataFrame({"Molecule Name": test["Molecule Name"].to_numpy()})

for endpoint in common.ENDPOINTS:
    y = train[endpoint]
    ok = y.notna().to_numpy()

    if ok.sum() < 50:
        print(f"  {endpoint}: only {int(ok.sum())} labels -- using the training mean")
        test_preds[endpoint] = y.mean()
        continue

    # The same reduction as above, refitted on every labelled molecule.
    sel = SelectKBest(f_regression, k=min(K_FEATURES, X_full.shape[1]))
    sel.set_output(transform="pandas")
    sel.fit(X_full[ok], y[ok])

    try:
        model = run_tabpfn(sel.transform(X_full[ok]), y[ok])
        test_preds[endpoint] = model.predict(sel.transform(X_test_clean))
        print(f"  {endpoint}: fitted on {int(ok.sum())} molecules")
    except Exception as exc:
        print(f"  {endpoint}: TabPFN failed ({type(exc).__name__}: {exc}) "
              "-- using the training mean")
        test_preds[endpoint] = y.mean()

test_preds.head()

In [ ]:
common.prepare_submission(test_preds, MODEL, SPLIT,
                          why=f"TabPFN on mordred descriptors, single task, k={K_FEATURES}")

To submit a **CheMeleon** model instead you need all nine endpoints, which
is eighteen training runs if you also want the from-scratch comparison. Pick the
endpoints where the gap was largest, use CheMeleon for those, and fill the rest
from a cheaper model &mdash; `Ensembles` has `per_endpoint_best` for exactly
this.